# Filtrage des espèces avec `ah ≠ ab`

Ce notebook parcourt le dossier `addmypet_by_model` (organisé par classe puis par modèle DEB)  
et identifie les espèces dont la valeur numérique de **`ah` (age at hatching)** est différente  
de celle de **`ab` (age at birth)**.

Pour chaque dossier `<class>/<model>` concerné, le notebook :
1. Copie les fichiers JSON filtrés dans un sous-dossier `ah_diff_ab/`
2. Met à jour le fichier `summary_no_class/summary_<class>_<model>.json` en ajoutant l'info en première clé

## Cellule 1 — Imports et configuration

In [5]:
import os
import json
import shutil

# Dossier source : organisé par addmypet_by_model/<class>/<model>/
addmypet_by_model = "../addmypet_by_model"

# Dossier contenant les fichiers summary par classe et modèle
summary_no_class = "../summary_no_class"

# Nom du sous-dossier à créer dans chaque dossier de modèle
subfolder_name = "ah_diff_ab"

## Cellule 2 — Fonction utilitaire : extraction de la valeur numérique

Certaines valeurs dans les JSON sont des expressions comme `C2K(25)` (température) — on ne retient que les tokens numériques purs.

In [6]:
def parse_value(val_str):
    """
    Extrait le premier token numérique d'une chaîne de valeur.
    Exemples : '9' -> 9.0  |  'C2K(25)' -> None  |  '1.5 d' -> 1.5
    Retourne None si la valeur est absente ou non numérique.
    """
    if val_str is None:
        return None
    try:
        return float(str(val_str).split()[0])
    except (ValueError, IndexError):
        return None

## Cellule 3 — Filtrage et copie des espèces avec `ah ≠ ab`

Pour chaque espèce (fichier JSON) dans chaque dossier `<class>/<model>` :
- On collecte toutes les valeurs numériques de `ah` et `ab`
- Si au moins une paire `(ah_val, ab_val)` est différente → l'espèce est sélectionnée
- Les fichiers sélectionnés sont copiés dans le sous-dossier `ah_diff_ab/`

In [7]:
total_copied = 0

# Dictionnaire pour garder en mémoire les résultats par (class, model)
# clé : (cls, model)  |  valeur : liste des noms de fichiers filtrés
results = {}

for cls in sorted(os.listdir(addmypet_by_model)):
    cls_path = os.path.join(addmypet_by_model, cls)
    if not os.path.isdir(cls_path):
        continue

    for model in sorted(os.listdir(cls_path)):
        model_path = os.path.join(cls_path, model)
        if not os.path.isdir(model_path):
            continue

        # Ignorer le sous-dossier de sortie lui-même
        if model == subfolder_name:
            continue

        species_with_ah_diff_ab = []

        for fname in os.listdir(model_path):
            if not fname.endswith(".json"):
                continue

            fpath = os.path.join(model_path, fname)
            try:
                with open(fpath) as f:
                    data = json.load(f)
            except Exception:
                continue

            # Collecter toutes les valeurs numériques de ah et ab pour cette espèce
            ah_vals, ab_vals = [], []
            for entry in data:
                if not isinstance(entry, dict):
                    continue
                entity = entry.get("entity")
                parsed = parse_value(entry.get("entity.value"))
                if parsed is None:
                    continue
                if entity == "ah":
                    ah_vals.append(parsed)
                elif entity == "ab":
                    ab_vals.append(parsed)

            # Sélectionner si l'espèce a les deux entités et qu'au moins une paire diffère
            if ah_vals and ab_vals:
                if any(ah != ab for ah in ah_vals for ab in ab_vals):
                    species_with_ah_diff_ab.append(fname)

        if not species_with_ah_diff_ab:
            continue

        # Créer le sous-dossier et y copier les fichiers filtrés
        out_dir = os.path.join(model_path, subfolder_name)
        os.makedirs(out_dir, exist_ok=True)
        for fname in species_with_ah_diff_ab:
            shutil.copy2(os.path.join(model_path, fname), os.path.join(out_dir, fname))

        results[(cls, model)] = species_with_ah_diff_ab
        total_copied += len(species_with_ah_diff_ab)
        print(f"  {cls}/{model}: {len(species_with_ah_diff_ab)} espèce(s) → {subfolder_name}/")

print(f"\nTotal espèces copiées : {total_copied}")

  Actinopterygii/abj: 20 espèce(s) → ah_diff_ab/
  Actinopterygii/std: 8 espèce(s) → ah_diff_ab/
  Amphibia/std: 4 espèce(s) → ah_diff_ab/
  Bivalvia/abj: 2 espèce(s) → ah_diff_ab/
  Clitellata/std: 1 espèce(s) → ah_diff_ab/
  Malacostraca/std: 1 espèce(s) → ah_diff_ab/
  Ophiuroidea/abj: 1 espèce(s) → ah_diff_ab/
  Reptilia/std: 1 espèce(s) → ah_diff_ab/

Total espèces copiées : 38


## Cellule 4 — Mise à jour des fichiers summary

Pour chaque `(class, model)` ayant des espèces filtrées, on ajoute une seule ligne  
`"has_species_with_ah_diff_ab": true` **juste après** la clé `total_species`  
dans `summary_no_class/summary_<class>_<model>.json`.

Si la clé est déjà présente (exécution précédente), le fichier est ignoré (**idempotent**).

In [8]:
summaries_updated = 0

for (cls, model), species_files in results.items():
    summary_fname = f"summary_{cls}_{model}.json"
    summary_path = os.path.join(summary_no_class, summary_fname)

    if not os.path.isfile(summary_path):
        print(f"  [WARNING] fichier non trouvé : {summary_fname}")
        continue

    try:
        with open(summary_path) as f:
            summary = json.load(f)
    except Exception as e:
        print(f"  [ERROR] lecture impossible de {summary_fname} : {e}")
        continue

    # Ne pas réécrire si la clé existe déjà (idempotent)
    if isinstance(summary, dict) and "has_species_with_ah_diff_ab" in summary:
        print(f"  [SKIP] {summary_fname} déjà mis à jour")
        continue

    # Insérer 'has_species_with_ah_diff_ab': True juste après 'total_species'
    updated_summary = {}
    for key, val in summary.items():
        updated_summary[key] = val
        if key == "total_species":
            updated_summary["has_species_with_ah_diff_ab"] = True

    with open(summary_path, "w") as f:
        json.dump(updated_summary, f, indent=4, ensure_ascii=False)

    summaries_updated += 1
    print(f"  -> {summary_fname} mis à jour")

print(f"\nSummaries mis à jour : {summaries_updated}")

  [SKIP] summary_Actinopterygii_abj.json déjà mis à jour
  [SKIP] summary_Actinopterygii_std.json déjà mis à jour
  [SKIP] summary_Amphibia_std.json déjà mis à jour
  [SKIP] summary_Bivalvia_abj.json déjà mis à jour
  [SKIP] summary_Clitellata_std.json déjà mis à jour
  [SKIP] summary_Malacostraca_std.json déjà mis à jour
  [SKIP] summary_Ophiuroidea_abj.json déjà mis à jour
  [SKIP] summary_Reptilia_std.json déjà mis à jour

Summaries mis à jour : 0
